# 基本構文：イベント列を一つずつ調べる

対応する本文は [Pythonの基本](../chapters/basics.tex) の章末課題である。ここでは、検出器が記録した4件のデータから、信号値を取り出し、平均と時刻差を求める。

コードセルを上から順にShift+Enterで実行する。最初の3問は画面上の計算だけで進められる。各セルの下に出た値を、直前・直後の説明にある値と比べる。

## 問1：まず1件の記録を読む

次のリストの各辞書は、1件の記録を表す。`detector` は検出器名、`time_ns` は記録時刻 [ns]、`signal` は信号値である。4件目の `None` は、信号値が得られなかったことを表す。

In [1]:
events = [
    {"detector": "A", "time_ns": 1000, "signal": 12.4},
    {"detector": "B", "time_ns": 1180, "signal": 9.8},
    {"detector": "A", "time_ns": 1300, "signal": 15.1},
    {"detector": "B", "time_ns": 1510, "signal": None},
]

In [2]:
print(events[0])
print(events[0]["signal"])

{'detector': 'A', 'time_ns': 1000, 'signal': 12.4}
12.4


最初の行にはAの記録全体、次の行には `12.4` が表示される。`events[0]` はリストの先頭、続く `["signal"]` はその記録の信号値を選ぶ。

### 4件を順に読む

`for event in events` は、4件を1件ずつ `event` という名前で取り出す。次のセルでは `12.4`、`9.8`、`15.1`、`None` が1行ずつ表示される。

In [3]:
for event in events:
    print(event["signal"])

12.4
9.8
15.1
None


### 信号値がある3件を表示する

`if` の条件に `is not None` を書くと、値がない記録を除ける。`if event["signal"]` だけでは有効な `0.0` も除いてしまうので、Noneかどうかを明示する。

In [4]:
for event in events:
    if event["signal"] is not None:
        print(event["signal"])

12.4
9.8
15.1


次は検出器名と時刻も表示する。最初の行が `A 1000 12.4` となり、全部で3行が表示されることを確認する。`print()` に値をコンマで区切って渡すと、空白で区切って表示される。

In [5]:
for event in events:
    if event["signal"] is not None:
        print(event["detector"], event["time_ns"], event["signal"])

A 1000 12.4
B 1180 9.8
A 1300 15.1


## 問2：Aの信号値の平均を求める

まず元の4件から、Aの有効な信号値だけを集める。`append()` はリストの末尾へ値を追加する。次のセルで `[12.4, 15.1]` が表示されれば、必要な2件を取り出せている。

In [6]:
a_signals = []
for event in events:
    if event["detector"] == "A" and event["signal"] is not None:
        a_signals.append(event["signal"])
print(a_signals)
print(sum(a_signals) / len(a_signals))

[12.4, 15.1]
13.75


合計 `27.5` を2件で割ると、平均は `13.75` になる。同じ処理をAとBで使えるよう、次に関数へまとめる。

関数の丸括弧で受け取る `events` と `detector` を引数、`return` で呼び出し元へ渡す平均を返り値という。有効値がない場合は平均を計算できないので、`ValueError` で知らせる。関数の先頭の三重引用符の文章はdocstringと呼び、使い方を書く。

In [7]:
def mean_signal(events, detector):
    """指定した検出器の有効信号の算術平均を返す。

    Args:
        events: detector と signal を持つ辞書の列。
        detector: 集計する検出器名。
    Returns:
        None を除いた信号値の算術平均。
    Raises:
        ValueError: 該当する有効値がない場合。
    """
    values = []
    for event in events:
        if event["detector"] == detector and event["signal"] is not None:
            values.append(event["signal"])
    if not values:
        raise ValueError("有効な信号値がありません。")
    return sum(values) / len(values)

print(mean_signal(events, "A"))
print(mean_signal(events, "B"))

13.75
9.8


### 答えと例外を自動で確認する

上の表示はAが `13.75`、Bが `9.8` になる。`assert 条件` は、条件を満たさないと `AssertionError` で止まる確認用の命令である。何もエラーが出なければ、その条件を満たしている。

Cの記録はないので、Cの平均を求めると `ValueError` が発生する。`try` と `except` では、予想した例外の名前と説明を表示する。

In [8]:
assert mean_signal(events, "A") == 13.75
assert mean_signal(events, "B") == 9.8
assert mean_signal([{"detector": "A", "signal": 0.0}], "A") == 0.0
try:
    mean_signal(events, "C")
except ValueError as exc:
    print(type(exc).__name__, str(exc))
else:
    raise AssertionError("有効値がない場合の例外が発生しませんでした。")

ValueError 有効な信号値がありません。


## 問3：隣り合う時刻の差を求める

まず4件の時刻をリストへ集める。`sorted()` は小さい順に並べたリストを返す。この入力では `[1000, 1180, 1300, 1510]` になる。

In [9]:
times = []
for event in events:
    times.append(event["time_ns"])
times = sorted(times)
print(times)
print(times[1] - times[0])

[1000, 1180, 1300, 1510]
180


最初の差は `1180 - 1000` で180 nsとなる。残りも同じように計算すると、`[180, 120, 210]` になる。4個の時刻の間は3か所ある。

`range(len(times) - 1)` は `0, 1, 2` を返す。最後の `i + 1` は3となり、リストの最後の時刻を取り出す。

In [10]:
gaps = []
for i in range(len(times) - 1):
    gaps.append(times[i + 1] - times[i])
print(gaps)
assert gaps == [180, 120, 210]

# 同じ取り出し方を、リスト内包表記で短く書ける。
compact_times = sorted([event["time_ns"] for event in events])
assert compact_times == times

[180, 120, 210]


## 問4：端末から条件を指定する

検出器Aのうち、信号値が13以上の記録を表示するプログラムを作る。保存した `filter_events.py` を端末で実行するときの入力は、次の形になる。

```sh
python filter_events.py A --min-signal 13
```

`A` は必須の位置引数、`--min-signal` は省略できるオプション引数である。信号の下限を省くと0.0を使う。ここからはファイルを作るので、保存先を用意する。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/03_basics/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [11]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "03_basics"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/03_basics


### ファイルへ書くコードを確認する

次のセルは、保存するコードを文字列として表示する。`argparse` は端末に入力された引数を読むための標準ライブラリである。`parse_args()` が読んだ検出器名と下限値を、最後の `if` の条件に使う。

In [12]:
program = "import argparse\n\nevents = " + repr(events) + "\n\n" + """parser = argparse.ArgumentParser()
parser.add_argument("detector")
parser.add_argument("--min-signal", type=float, default=0.0)
args = parser.parse_args()
for event in events:
    if (event["detector"] == args.detector
            and event["signal"] is not None
            and event["signal"] >= args.min_signal):
        print(event["detector"], event["time_ns"], event["signal"])
"""
print(program)

import argparse

events = [{'detector': 'A', 'time_ns': 1000, 'signal': 12.4}, {'detector': 'B', 'time_ns': 1180, 'signal': 9.8}, {'detector': 'A', 'time_ns': 1300, 'signal': 15.1}, {'detector': 'B', 'time_ns': 1510, 'signal': None}]

parser = argparse.ArgumentParser()
parser.add_argument("detector")
parser.add_argument("--min-signal", type=float, default=0.0)
args = parser.parse_args()
for event in events:
    if (event["detector"] == args.detector
            and event["signal"] is not None
            and event["signal"] >= args.min_signal):
        print(event["detector"], event["time_ns"], event["signal"])



次のセルでファイルを保存し、別のPythonとして起動する。`sys.executable` は、このNotebookを実行しているPythonの場所を表す。結果が `A 1300 15.1` の1行になることを確認する。

In [13]:
import subprocess

script = WORK / "filter_events.py"
script.write_text(program, encoding="utf-8")
result = subprocess.run([sys.executable, str(script), "A", "--min-signal", "13"],
                        capture_output=True, text=True, check=True)
print(result.stdout)
assert result.stdout.strip() == "A 1300 15.1"

A 1300 15.1

